# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show all record sets and their @id
print("Record Sets and their fields:")
record_sets = dataset.record_sets  # list of mlcroissant.RecordSet

all_recordsets = []
for rs in record_sets:
    print(f"- RecordSet: {rs.id}")
    all_recordsets.append(rs.id)
    for field in rs.fields:
        print(f"    - Field: {field.id} ({field.name}), dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

# For demonstration, use the first record set present
if len(record_sets) == 0:
    raise Exception('No record sets found in this Croissant schema.')
primary_record_set_id = record_sets[0].id
record_sets_ids = [rs.id for rs in record_sets]

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print(f"Fields (columns) available in {primary_record_set_id}:")
print(dataframes[primary_record_set_id].columns.tolist())
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll choose a numeric field for the analysis
# First, list all numeric (float or int) fields in the chosen record set
df = dataframes[primary_record_set_id]
numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
print(f"Numeric fields: {numeric_cols}")

# If there are no numeric fields, raise a warning
if not numeric_cols:
    raise Exception("No numeric fields found for analysis.")

# Use the first numeric field for demonstration
numeric_field = numeric_cols[0]
print(f"Selected numeric field for filtering and normalization: {numeric_field}")

# Filter values
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records in {primary_record_set_id} with {numeric_field} > mean [{threshold:.2f}]: {len(filtered_df)} rows")
print(filtered_df[[numeric_field]].head())

# Normalize field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt grouping by another field, e.g., by protein description or first string column that is not the unique id
string_cols = df.select_dtypes(include=['object']).columns.tolist()
group_field = None
for col in string_cols:
    if col != numeric_field:
        group_field = col
        break
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (showing average {numeric_field} per {group_field}):")
    print(grouped_df.head())
else:
    print("No suitable string field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot histogram of the selected numeric field
plt.figure(figsize=(8, 5))
df[numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouping field is available, plot top groups by mean
if group_field is not None:
    top_grouped = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    plt.bar(top_grouped[group_field], top_grouped[numeric_field])
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Top 10 {group_field} by average {numeric_field}")
    plt.ylabel(f"Average {numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded the mass spectrometry dataset via its Croissant schema with `mlcroissant`.
- Explored available record sets and fields using their unique `@id`s.
- Loaded the primary record set as a DataFrame and examined numeric fields.
- Performed basic filtering and normalization on a selected numeric attribute.
- Grouped data by a categorical field for aggregate analysis.
- Visualized data distributions using histograms and bar charts.

This process demonstrates how to apply FAIR dataset standards and the `mlcroissant` library for initial data understanding and preparation prior to downstream analytics or machine learning.